In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-8B"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

# Change this prompt to test different capabilities
prompt = """
Complete these tasks:

1. Translate this Vietnamese sentence into English:
   "Bệnh nhân bị tăng huyết áp và đang sử dụng amlodipine."

2. Extract diseases, symptoms, and medications as valid JSON.

3. Write a Python function that checks whether a number is prime.

Keep your answer concise.
"""

messages = [
    {
        "role": "system",
        "content": "You are a precise and helpful AI assistant."
    },
    {
        "role": "user",
        "content": prompt
    }
]

# Convert chat messages into Qwen's required prompt format
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False  # Change to True to test reasoning mode
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20
    )

# Remove the original input tokens and keep only Qwen's response
response_ids = generated_ids[0, inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    response_ids,
    skip_special_tokens=True
)

print("\n========== QWEN RESPONSE ==========\n")
print(response)

c:\Users\ACER\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ACER\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--Qwen--Qwen3-8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator


========== QWEN RESPONSE ==========

1. "The patient has hypertension and is using amlodipine."

2.  
```json
{
  "diseases": ["hypertension"],
  "medications": ["amlodipine"]
}
```

3.  
```python
def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
```
